# Chaîne OSF CAPC-AC — Notebook 05 · La calibration, produit par produit (phase P3)

**Objectif :** comprendre *pourquoi* chaque produit reçoit une méthode différente, et savoir
lire un gain sans se tromper.

La phase P2 a établi le diagnostic : tous les modèles **ordonnent** correctement les années
(GROC entre 0,52 et 0,60 partout) mais **aucun** ne livre des valeurs ni des probabilités
utilisables telles quelles (MSESS du cumul entre −1 et −6, RPSS ≈ 0). C'est le cas d'école de la
surconfiance corrigible : il y a du signal, il est mal exprimé. La calibration est l'outil qui
corrige l'expression sans inventer de signal — et ce notebook montre où elle réussit, où elle
échoue, et pourquoi.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
cfg = load_cycle("config/cycle_202609.yaml")
OUT = cfg.output_root / "skill" / cfg.cycle_id / "calibrated"
REG = cfg.path_of("output_root") / "registry"
registre = pd.read_csv(REG / "calibration_methods.csv")
comparaison = pd.read_csv(OUT / "comparison_raw_calibrated.csv")
print(f"{len(registre)} décisions au registre, {len(comparaison)} comparaisons")


## 1. Une méthode par question, pas une méthode pour tout

La note de méthode du CAPC-AC pose la règle : *chaque indicateur est une question opérationnelle
différente, donc il a sa méthode*. Cela se lit dans les deux sens — une méthode n'est pas
essayée sur un produit auquel elle ne peut pas répondre, et un produit n'est pas jugé sur une
méthode qui ne le visait pas.

Trois choses décident :

1. **ce que la méthode produit.** Une régression logistique par catégorie produit trois
   probabilités et rien d'autre : elle ne peut pas rendre un cumul en millimètres, ni répondre à
   « plus de 200 mm ». Une ELR, un NGR/EMOS ou un ensemble corrigé produisent une **distribution
   complète** : ils répondent à tout ;
2. **ce que le produit demande à la distribution.** Une valeur lit la moyenne, trois catégories
   lisent deux seuils, un événement en lit un ;
3. **si le produit est défini par un quantile ou par une valeur.** Un produit en millimètres est
   celui qui a le plus besoin que la distribution soit posée sur l'échelle de l'observation.

Le tableau ci-dessous est la table du module `calibrate/products.py` : elle se lit, se discute
et se modifie comme une table.

In [ ]:
from eccas_s2s.calibrate.products import method_product_table
from eccas_s2s.operations.calibrate_hindcast import calibrators_for
from eccas_s2s.products.catalogue import catalogue

methodes = calibrators_for("precip", has_members=True)
produits = catalogue("precip", "season", cfg.thresholds.get("precip", {}))
table = pd.DataFrame(method_product_table(produits, methodes))
vue = table.pivot_table(index=["product", "deciding_metric"], columns="method",
                        values="applicable", aggfunc="first")
display(vue[[c.name for c in methodes]].replace({True: "x", False: "·"}))


## 2. Lire un gain sans se tromper

Le piège est la **médiane**. Une méthode qui répare la bande équatoriale et casse le Sahel a le
même gain médian qu'une méthode qui ne fait rien. C'est pourquoi le gain est calculé **en chaque
maille** et résumé par quatre nombres :

| | |
|---|---|
| `median_gain` | le gain de la maille typique |
| `fraction_improved` | la part du masque où le gain est positif |
| `fraction_repaired` | la part où le produit **devient** utilisable (le brut était sous sa valeur sans skill, le calibré est au-dessus) |
| `fraction_broken` | la part où il **cesse** de l'être |

La cellule suivante le montre sur un cas fabriqué : deux méthodes de gain médian nul, qui n'ont
rien à voir.

In [ ]:
from eccas_s2s.validate.comparison import compare_maps

coords = {"latitude": np.linspace(-4, 4, 20), "longitude": np.linspace(10, 20, 20)}
brut = xr.DataArray(np.full((20, 20), -0.10), dims=("latitude", "longitude"), coords=coords)
masque = xr.full_like(brut, True, dtype=bool)

inerte = brut.copy()                                   # ne change rien
mixte = brut.copy(); mixte[:10, :] = 0.10; mixte[10:, :] = -0.30   # répare la moitié, casse l'autre

lignes = [compare_maps(inerte, brut, "rpss", masque, method="inerte", product="terciles"),
          compare_maps(mixte, brut, "rpss", masque, method="mixte", product="terciles")]
display(pd.DataFrame(lignes)[["method", "median_gain", "fraction_improved",
                              "fraction_repaired", "fraction_broken"]].round(3))


Gain médian identique (nul), conclusions opposées : la méthode « mixte » rend le produit
utilisable sur la moitié du domaine et ne casse rien (le brut y était déjà négatif), la méthode
« inerte » ne fait rien du tout. Une recommandation fondée sur la seule médiane les
confondrait.

## 3. Ce que la calibration a réellement apporté

Sur les 8 modèles, mois et saisons. Deux lectures s'imposent, et elles disent des choses
différentes.

In [ ]:
keys = ["system", "model", "variable", "scale", "product", "metric", "method"]
retenue = comparaison.merge(registre[keys], on=keys)
bilan = retenue.groupby("product").agg(brut=("median_raw", "median"),
                                       calibre=("median_calibrated", "median"))
bilan["gain"] = (bilan["calibre"] - bilan["brut"])
bilan["utilisable_brut_%"] = retenue.groupby("product")["median_raw"].apply(lambda s: 100*(s > 0).mean())
bilan["utilisable_calibre_%"] = retenue.groupby("product")["median_calibrated"].apply(lambda s: 100*(s > 0).mean())
display(bilan.round(3))


**Première lecture — elle efface le biais.** Le MSESS du cumul passe de −2,07 à −0,13, le
score de Brier du seuil 600 mm de −0,62 à −0,09. Les produits en valeur absolue, inutilisables
bruts, redeviennent proches de la climatologie. C'est considérable, et c'était l'essentiel du
problème diagnostiqué en P2.

**Seconde lecture — elle n'invente pas de signal.** Même calibré, le cumul saisonnier ne bat pas
la climatologie au point typique. Ce qui progresse vraiment en *utilisabilité*, ce sont les
produits probabilistes lus en quantiles : les terciles passent de 6 % à 28 % des cas où la maille
typique a un RPSS positif, les classes SPI de 97 % à 100 %.

Les deux lectures sont vraies en même temps, et c'est exactement ce qu'un registre honnête doit
dire.

## 4. Pourquoi l'anomalie et le SPI restent bruts

Dans les 16 cas sur 16, aucune méthode ne les améliore, et le registre recommande `raw`. Ce
n'est pas un échec de la calibration, c'est une propriété des deux produits :

* l'**anomalie** est jugée sur l'ACC, une corrélation — insensible à un biais additif **et** à un
  facteur multiplicatif. Or c'est précisément ce que corrigent les corrections de biais, l'EQM et
  le NGR. Il n'y a rien à gagner ;
* le **SPI** est calculé dans la distribution du modèle lui-même (gamma ajustée sur son propre
  hindcast, comme dans `run_forecast_v2`) : il est normalisé par construction, donc déjà exempt
  du biais que la calibration retire.

Le registre l'écrit plutôt que d'imposer une méthode inutile — et le prévisionniste sait que ces
deux produits se publient tels quels.

In [ ]:
brut_partout = registre[registre.method == "raw"]["product"].value_counts()
print("produits laissés bruts (nombre de modèles × échelles) :")
print(brut_partout.to_string())


## 5. Quelle méthode gagne, et où

Le profil n'est pas aléatoire : **ELR** domine les seuils (elle modélise la distribution seuil
par seuil, ce que demande « plus de 300 mm »), **NGR** domine les terciles et le cumul (elle
corrige la moyenne *et* la dispersion, donc la surconfiance), les corrections de biais simples
ne l'emportent que ponctuellement.

In [ ]:
display(registre.groupby(["product"])["method"].value_counts().unstack(fill_value=0))


## 6. Le modèle le plus aidé est le plus faible

DWD, qui avait le RPSS brut le plus bas (−0,128 en saison), gagne le plus (+0,092) ; Météo-France,
déjà proche de zéro, ne gagne presque rien. La calibration rattrape la surconfiance là où elle
est la plus forte — ce que la phase P2 laissait prévoir avec son GROC positif partout.

In [ ]:
terciles = retenue[(retenue["product"] == "terciles") & (retenue.scale == "season")]
display(terciles.groupby("model")[["median_raw", "median_calibrated", "median_gain",
                                   "fraction_improved", "fraction_repaired"]].mean().round(3))


## 7. La carte de gain : *où* la calibration agit

Les quatre nombres résument ; la carte montre. Vert : le score s'améliore en cette maille ;
brun : il se dégrade. Les cartes sont lissées entre les mailles (ombrage de Gouraud) puis
découpées exactement sur le contour CEEAC — un score varie continûment dans l'espace, et une
carte en damier se lirait comme si chaque maille était une mesure indépendante.

In [ ]:
from IPython.display import Image, display
from eccas_s2s.operations.skill_raw import score_paths

model, scale, periode = "dwd", "season", "season_m1"
methode = registre[(registre.model == model) & (registre.scale == scale)
                   & (registre["product"] == "terciles")]["method"].iat[0]
for quoi in (f"{methode}/rpss", f"{methode}/gain_rpss"):
    f = score_paths(OUT / "figures", "c3s", model, "precip", scale, "terciles") / quoi / f"{periode}.png"
    if f.exists():
        display(Image(filename=str(f), width=560))


## Ce qu'il faut retenir

1. **Chaque produit a sa méthode**, et une méthode n'est pas essayée là où elle ne peut pas
   répondre : la logistique par catégorie ne rend pas un cumul, l'ELR ne rend pas une valeur.
2. **Le gain se mesure en chaque maille.** Quatre nombres — gain médian, surface améliorée,
   réparée, cassée — parce qu'une médiane seule confond « répare l'équateur et casse le Sahel »
   avec « ne fait rien ».
3. La référence du gain est le **brut interpolé sur la grille d'observation**, jamais les cartes
   de P2 calculées à 1° : sinon on mesurerait aussi le changement de grille.
4. La calibration **efface le biais** (cumul : MSESS −2,07 → −0,13) mais **n'invente pas de
   signal** : le cumul saisonnier reste sous la climatologie au point typique.
5. **Anomalie et SPI restent bruts** : insensibles par construction à ce que la calibration
   corrige. Le registre le dit au lieu de l'ignorer.
6. Le modèle le plus aidé est le plus faible — la calibration rattrape la surconfiance, elle ne
   remplace pas la prévisibilité.
